In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")

model_name = "textattack/distilbert-base-uncased-MRPC"
batch_size = 64
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Max length: {max_length}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
def tokenize_batch(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=max_length
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_dataset.set_format(type="torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
dataloader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

print(f"Prepared DataLoader with {len(dataloader)} batches.")


In [ ]:
predictions = []
true_labels = []

for batch in dataloader:
    labels = batch["label"]
    inputs = {k: v.to(device) for k, v in batch.items() if k != "label"}

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu()

    predictions.extend(preds.tolist())
    true_labels.extend(labels.tolist())

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": len(dataset),
        "batch_size": batch_size,
        "max_length": max_length,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device)
    }
])

print(results_df.to_string(index=False))
print("Confusion Matrix:")
print(cm)


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df[~examples_df["correct"]].copy()
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    sample_errors_df = mismatches_df.head(10)
    print(sample_errors_df[["sentence1", "sentence2", "true_label", "predicted_label"]].to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
